<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 35px; border-radius: 12px; color: white; margin-bottom: 25px;">
  <h1 style="margin: 0 0 8px 0; font-size: 2em;">🏦 Lab 01: Environment Setup &amp; Data Exploration</h1>
  <p style="margin: 0; font-size: 1.15em; opacity: 0.92;">Zava Bank Workshop — Microsoft Foundry Agent Observability</p>
</div>

<div style="background: #f5f5f5; border-left: 4px solid #533483; padding: 16px 20px; border-radius: 6px; margin-bottom: 20px;">

## What We Learn

- **Validate** the Azure AI Projects SDK and supporting packages are installed
- **Load** the Zava Bank financial data — client portfolios, market indicators, and risk metrics
- **Preview** client portfolios to understand the advisory dataset before building agents

</div>

## 📖 Zava Bank Story

Zava Bank's wealth-advisory team is overwhelmed. Every advisor manages dozens of high-net-worth clients, each with a unique portfolio, risk tolerance, and set of questions — *"How is my tech exposure performing?"*, *"Am I over-concentrated?"*, *"What does the macro outlook mean for my bonds?"*

Answering these questions today requires an advisor to manually pull portfolio data, cross-reference market indicators, review risk metrics, and synthesize it all into a coherent response. The team needs an **AI-powered advisory assistant** that can:

1. **Search portfolios** — look up any client's holdings, sector allocations, and account types
2. **Assess risk** — surface beta, Sharpe ratio, max drawdown, and concentration warnings
3. **Pull market data** — retrieve current index levels, sector performance, and economic indicators

This lab sets up the development environment and explores the data that will power our agent in the labs ahead.

---
## 1 · Install &amp; Validate SDK

In [ ]:
# Install / upgrade the core SDK — quiet mode keeps output clean
%pip install azure-ai-projects azure-identity openai python-dotenv pandas --quiet

In [ ]:
import importlib, sys

required = [
    "azure.ai.projects",
    "azure.identity",
    "openai",
    "pandas",
    "dotenv",
]

for mod in required:
    try:
        importlib.import_module(mod)
        print(f"  ✅ {mod}")
    except ImportError:
        print(f"  ❌ {mod} — run the pip install cell above")

print(f"\nPython {sys.version}")

---
## 2 · Load Environment &amp; Create Clients

<div style="background: #f5f5f5; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Before running this cell** — make sure you have copied `sample.env` → `.env` in the `notebooks/` directory and filled in your project endpoint and model deployment name.

</div>

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Walk up from the notebook directory to find the .env
notebook_dir = Path.cwd()
env_path = notebook_dir
while env_path != env_path.parent:
    if (env_path / ".env").exists():
        break
    env_path = env_path.parent

load_dotenv(env_path / ".env", override=True)

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

print(f"Project endpoint : {PROJECT_ENDPOINT[:60]}..." if len(PROJECT_ENDPOINT) > 60 else f"Project endpoint : {PROJECT_ENDPOINT}")
print(f"Model deployment : {MODEL_DEPLOYMENT}")

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
print("✅ AIProjectClient ready")

# The OpenAI-compatible client is used in later labs for direct model calls
openai_client = project_client.inference.get_azure_openai_client(api_version="2025-03-01-preview")
print("✅ OpenAI client ready")

---
## 3 · Load Zava Bank Data

The advisory agent relies on three CSV files that simulate Zava Bank's internal data systems:

| File | Description |
|---|---|
| `client_portfolios.csv` | Holdings per client — ticker, shares, cost basis, current price, sector, account type |
| `market_data.csv` | Market indices, sector performance, and economic indicators |
| `risk_metrics.csv` | Per-client risk profile — beta, Sharpe ratio, max drawdown, sector exposures |

In [ ]:
import pandas as pd

DATA_DIR = notebook_dir.parents[1] / "data" / "zava-bank"

portfolios_df = pd.read_csv(DATA_DIR / "client_portfolios.csv")
market_df     = pd.read_csv(DATA_DIR / "market_data.csv")
risk_df       = pd.read_csv(DATA_DIR / "risk_metrics.csv")

print(f"✅ Loaded from {DATA_DIR}")
print(f"   Portfolios  : {len(portfolios_df):>3} rows")
print(f"   Market data : {len(market_df):>3} rows")
print(f"   Risk metrics: {len(risk_df):>3} rows")

---
### 3a · Portfolio Summary

In [ ]:
total_aum = portfolios_df["market_value"].sum()
num_clients = portfolios_df["client_id"].nunique()
positions_per_client = portfolios_df.groupby("client_id").size()

print("── Portfolio Summary ──────────────────────────────")
print(f"   Total AUM          : ${total_aum:,.2f}")
print(f"   Number of clients  : {num_clients}")
print(f"   Positions / client : {positions_per_client.min()} – {positions_per_client.max()} (avg {positions_per_client.mean():.1f})")
print()

# AUM by client
client_aum = (
    portfolios_df.groupby(["client_id", "client_name"])["market_value"]
    .sum()
    .reset_index()
    .sort_values("market_value", ascending=False)
)
client_aum["pct_of_total"] = (client_aum["market_value"] / total_aum * 100).round(1)
client_aum

In [ ]:
# Sector allocation across all clients
sector_alloc = (
    portfolios_df.groupby("sector")["market_value"]
    .sum()
    .sort_values(ascending=False)
)
sector_alloc_pct = (sector_alloc / total_aum * 100).round(1)

print("── Sector Allocation (all clients) ────────────────")
for sector, pct in sector_alloc_pct.items():
    bar = "█" * int(pct / 2)
    print(f"   {sector:<20} {pct:>5.1f}%  {bar}")

---
### 3b · Market Data Summary

In [ ]:
print("── Market Indices & Economic Indicators ───────────")
print(f"   Total indicators : {len(market_df)}")
print(f"   Categories       : {', '.join(market_df['category'].unique())}")
print(f"   As of            : {market_df['date'].iloc[0]}")
print()
market_df[["indicator_id", "name", "category", "current_value", "change_pct"]]

---
### 3c · Risk Metrics Summary

In [ ]:
print("── Client Risk Profiles ───────────────────────────")
print(f"   Clients profiled : {len(risk_df)}")
print(f"   Risk tolerances  : {', '.join(risk_df['risk_tolerance'].unique())}")
print()
risk_df[["client_id", "client_name", "risk_tolerance", "portfolio_beta", "sharpe_ratio", "max_drawdown_pct", "concentration_top3_pct", "total_value"]]

---
## 4 · Data Quality Check

Before building an agent on top of this data, we verify **referential integrity** — every client in the portfolio file should have a matching risk profile, and vice versa.

In [ ]:
portfolio_clients = set(portfolios_df["client_id"].unique())
risk_clients      = set(risk_df["client_id"].unique())

in_portfolio_not_risk = portfolio_clients - risk_clients
in_risk_not_portfolio = risk_clients - portfolio_clients

print("── Referential Integrity ──────────────────────────")
print(f"   Clients in portfolios : {len(portfolio_clients)}")
print(f"   Clients in risk file  : {len(risk_clients)}")
print()

if not in_portfolio_not_risk and not in_risk_not_portfolio:
    print("   ✅ All client IDs match between portfolios and risk metrics")
else:
    if in_portfolio_not_risk:
        print(f"   ⚠️  In portfolios but missing from risk: {in_portfolio_not_risk}")
    if in_risk_not_portfolio:
        print(f"   ⚠️  In risk but missing from portfolios: {in_risk_not_portfolio}")

# Verify that total_value in risk_df is consistent with summed market_value
print()
print("── AUM Consistency ────────────────────────────────")
for _, row in risk_df.iterrows():
    calc_value = portfolios_df.loc[
        portfolios_df["client_id"] == row["client_id"], "market_value"
    ].sum()
    match = "✅" if abs(calc_value - row["total_value"]) < 0.01 else "⚠️"
    print(f"   {match} {row['client_name']:<25} risk_file=${row['total_value']:>12,.2f}  calculated=${calc_value:>12,.2f}")

# Verify sectors in portfolios align with market data sectors
portfolio_sectors = set(portfolios_df["sector"].unique())
market_sectors    = set(
    market_df.loc[market_df["category"] == "Sector Performance", "name"]
    .str.replace(" Sector", "")
)

print()
print("── Sector Coverage ────────────────────────────────")
print(f"   Portfolio sectors : {sorted(portfolio_sectors)}")
print(f"   Market sectors    : {sorted(market_sectors)}")
unmapped = portfolio_sectors - market_sectors
if not unmapped:
    print("   ✅ All portfolio sectors have market data")
else:
    print(f"   ⚠️  Sectors without market data: {unmapped}")

---

<div style="background: linear-gradient(135deg, #0f3460, #533483); padding: 20px 25px; border-radius: 10px; color: white; margin-top: 25px;">

## ✅ Lab 01 Summary

Environment is configured and all three Zava Bank datasets are loaded and validated.

**Key numbers to keep in mind:**
- **5 clients** ranging from conservative individual accounts to an aggressive institutional fund
- **29 portfolio positions** spanning 7 sectors
- **18 market indicators** covering indices, sector performance, and economic data
- **Total AUM** across all clients: ~$2.16M

**Next → Lab 02**: Create your first Client Advisor Agent using Foundry prompt agents.

</div>